# Validate Neo4j MCP Connection

Run these cells after completing `MCP-MANUAL-SETUP.md` to confirm the Unity Catalog HTTP connection is live and routing correctly to the Neo4j MCP server.

The three steps check connectivity in order: list tools, fetch schema, execute a Cypher query.

In [ ]:
# Unity Catalog HTTP connection created in MCP-MANUAL-SETUP.md
MCP_CONNECTION  = "neo4j_aircraft_mcp_server"

# AgentCore Gateway prefixes tool names with its target id.
# Run Step 1 (tools/list) first to confirm the exact names for your deployment.
TOOL_GET_SCHEMA  = "neo4j-mcp-server-target___get_neo4j_schema"
TOOL_READ_CYPHER = "neo4j-mcp-server-target___read_neo4j_cypher"

print(f"Connection: {MCP_CONNECTION}")

In [ ]:
import json


def _post(payload: str) -> tuple[int, str]:
    row = spark.sql(
        f"""
        SELECT http_request(
          conn    => '{MCP_CONNECTION}',
          method  => 'POST',
          path    => '',
          headers => map('Content-Type', 'application/json'),
          json    => :payload
        ) AS r
        """,
        args={"payload": payload},
    ).collect()[0]["r"]
    return row["status_code"], row["text"]


def _parse(text: str) -> dict:
    for line in text.splitlines():
        stripped = line.strip()
        if stripped.startswith("data:"):
            text = stripped[len("data:"):].strip()
            break
    return json.loads(text)


def mcp_call(method: str, params: dict | None = None) -> dict:
    status, text = _post(
        json.dumps({"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}})
    )
    if status != 200:
        raise RuntimeError(f"HTTP {status}: {text}")
    body = _parse(text)
    if "error" in body:
        raise RuntimeError(f"MCP error: {body['error']}")
    return body.get("result", {})


def read_cypher(query: str) -> str:
    result = mcp_call("tools/call", {"name": TOOL_READ_CYPHER, "arguments": {"query": query}})
    return "\n".join(p.get("text", "") for p in result.get("content", []))


print("Helpers ready.")

## Step 1: List tools

Confirms the connection is reachable and the MCP flag is active. If your tool names differ from the defaults in the config cell, update `TOOL_GET_SCHEMA` and `TOOL_READ_CYPHER` to match.

In [ ]:
result = mcp_call("tools/list")
tools  = result.get("tools", [])
print(f"{len(tools)} tool(s) available:\n")
for t in tools:
    print(f"  {t['name']}")
    desc = (t.get("description") or "").strip().splitlines()
    if desc:
        print(f"    {desc[0][:100]}")

## Step 2: Get schema

Retrieves node labels, relationship types, and properties from Neo4j.

In [ ]:
result = mcp_call("tools/call", {"name": TOOL_GET_SCHEMA, "arguments": {}})
schema = "\n".join(p.get("text", "") for p in result.get("content", []))
print(schema)

## Step 3: Run a Cypher query

Confirms `read_neo4j_cypher` routes correctly to Neo4j and returns results.

In [ ]:
print(read_cypher("""
    MATCH (n)
    WITH labels(n) AS lbls
    UNWIND lbls AS label
    RETURN label, count(*) AS count
    ORDER BY count DESC
"""))